# Notebook pour l'écriture du TEIheader

Les `teiHeader` seront généré à partir des réponses du [form](https://docs.google.com/forms/d/e/1FAIpQLSczVG3VSlDDcI403e02jDjFJyUmcxvRhKmNePzl_TwLIC-9xQ/viewform?usp=**sharing**) mais certaines informations sont nécessaires pour que le lien soit fait avec les index qui sont nécessaire pour la partie indexation et que le document xml soit validé par Oxygen.  
On copie les informations du tableau dans le fichier .csv sans modifier l'ordre des colonnes et sans prendre en compte la première colonne "horodateur".  
Le résultat du header au plus complet est indiqué à titre d'exemple dans le document [header-front.xml] sur le dépôt Git. 

## Chargement des bibliothèques

In [32]:
import csv
import xml.etree.ElementTree as ET

## Lecture du CSV

Le fichier CSV associé est `header.csv` qui est dans le même dossier.

La première colonne doit être :
``` cvs

NomAuteur,PrenomAuteur,FicheISNI,DBI/ARK,TitreOuvrage,LieuEdition,Editeur,Date,USTC,BiblioConservation,PaysConservation,VilleConservation,LienReproduction,Droits,EditionsModernes,Traductions,BibliographieSelective,Licence,ContributeursEmmaBondoerffer,ContributeursAnaïsMazoué,ContributeursTeklia,ContributeursHippolyteFaré,ContributeursCortoRolaz,ContributeursOctavePhilibert,ContributeursAlessiaAvena,ContributeursYoussoufDiawara,ContributeursJuliaCastiglione,ContributeursAnnaSconza,NiveauTranscription,SpecificitesDuXML,StructureFichierXML

```

Si jamais elle est supprimée dans le fichier header, il faut la recopier comme ci-dessus pour que la fonction de lecture du CSV reconnaisse les colonnes.


In [33]:
def lire_csv_header(fichier_header): #création d'une fonction pour lire les csv
    infos_header = [] #création d'une liste vide pour charger les données du csv
    try:
        with open(fichier_header, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                # Vérification des colonnes nécessaires
                if row.get('Titre') and row.get('Nom'): 
                    infos_header.append({
                        'Nom': row['Nom'].strip(),
                        'Prenom': row['Prenom'].strip(),
                        'ISNI': row['ISNI'].strip(),
                        'Titre': row['Titre'].strip(),
                        'pubPlace': row['Lieu'].strip(),
                        'Editeur': row['Editeur'].strip(),
                        'Date': row['Annee'].strip(),
                        'LienReproduction': row['LienEdRef'].strip(),
                        'Droits': row.get('Droits', '').strip(),
                        'Emma Bondoerffer': row.get('ContributeursEmmaBondoerffer', '').strip(),
                        'Anaïs Mazoué': row.get('ContributeursAnaïsMazoué', '').strip(),
                        'Teklia': row.get('ContributeursTeklia', '').strip(),
                        'Julia Castiglione': row.get('ContributeursJuliaCastiglione', '').strip(),
                        'Irene Calcagno': row.get('ContributeursIreneCalcagno', '').strip(),
                        'Memofonte': row.get('ContributeursMemofonte', '').strip(),
                        'ATIR': row.get('ContributeursATIR', '').strip()
                    })
                else:
                    print(f"Ligne ignorée (champs manquants) : {row}") 
    except Exception as e: # il faut mettre un except quand on met try, permet de signaler les erreurs
        print(f"Erreur lors de la lecture du fichier CSV : {e}")
    print(infos_header)
    return infos_header
    

Cette cellule n'est pas nécessaire mais elle permet de vérifier le résultat de la fonction lire_csv_header

In [34]:
# Vérification de la liste crée avec les infos du csv.
fichier_header = 'header_synth.csv'
lire_csv_header(fichier_header)

[{'Nom': 'Armenini', 'Prenom': 'Giovan Battista', 'ISNI': 'https://isni.org/isni/0000000118388240', 'Titre': "De' veri precetti della pittura di M. Gio. Battista Armenini da Faenza libri tre: Ne' quali con bell' ordine d' utili, & buoni avertimenti, per chi desidera in essa farsi con prestezza eccellente; si dimostrano i modi principali del disegnare, & del dipingere, & de fare le Pitture, che si convengono alle conditioni de' luoghi, & delle persone. Opera non solo utile, & necessaria à tutti gli Artefici per cagion del disegno; lume, & fondamento di tutte l' altre arti minori, ma anco à ciascun' altra persona intendente di cosi nobili professione. Al Sereniss. Sig. il Signor Guglielmo Gonzaga Duca di Mantova, di Monferrato, &c. (Ravenna: Francesco Tebaldini, 1586)", 'pubPlace': 'Ravenna', 'Editeur': 'Francesco Tebaldini', 'Date': '1586', 'LienReproduction': 'https://www.digitale-sammlungen.de/en/view/bsb10151942?page=16,17', 'Droits': 'CC', 'Emma Bondoerffer': 'Structuration', 'Anaïs

[{'Nom': 'Armenini',
  'Prenom': 'Giovan Battista',
  'ISNI': 'https://isni.org/isni/0000000118388240',
  'Titre': "De' veri precetti della pittura di M. Gio. Battista Armenini da Faenza libri tre: Ne' quali con bell' ordine d' utili, & buoni avertimenti, per chi desidera in essa farsi con prestezza eccellente; si dimostrano i modi principali del disegnare, & del dipingere, & de fare le Pitture, che si convengono alle conditioni de' luoghi, & delle persone. Opera non solo utile, & necessaria à tutti gli Artefici per cagion del disegno; lume, & fondamento di tutte l' altre arti minori, ma anco à ciascun' altra persona intendente di cosi nobili professione. Al Sereniss. Sig. il Signor Guglielmo Gonzaga Duca di Mantova, di Monferrato, &c. (Ravenna: Francesco Tebaldini, 1586)",
  'pubPlace': 'Ravenna',
  'Editeur': 'Francesco Tebaldini',
  'Date': '1586',
  'LienReproduction': 'https://www.digitale-sammlungen.de/en/view/bsb10151942?page=16,17',
  'Droits': 'CC',
  'Emma Bondoerffer': 'Stru

## Générer le header

In [35]:
def generer_header(fichier_header, fichier_sortie): # définition de la fonction qui génère les headers
    header = lire_csv_header(fichier_header) # Utilise la liste créée avec les infos du csv

    teiHeader = ET.Element("listHeaders")
    for info in header : # Pour chaque ligne du csv, chaque élément de la liste, on fait un header tel que
        root = ET.SubElement(teiHeader, "teiHeader") # racine du XML 

# construction du XML en partant de la racine, enr ajoutant les éléments du header, les attributs et le contenu récupéré dans le CSV
        fileDesc_elem = ET.SubElement(root, "fileDesc") 
        titleStmt_elem = ET.SubElement(fileDesc_elem, "titleStmt")
        title_elem = ET.SubElement(titleStmt_elem, "title", attrib={"xml:lang": "fra"})
        title_elem.text = info["Titre"]
        
        for key in list(info.keys())[9:] : # pour les dernières lignes (attribution des resp) on fait une boucle pour que 
            # si le contenenu est non-vide on crée une entrée dans le header en utilisant le nom de la colonne et le contenu pour donner la resp.
            if info[key] != '' :
                respStmt_elem = ET.SubElement(titleStmt_elem, "respStmt")
                name_elem = ET.SubElement(respStmt_elem, "name")
                name_elem.text = key
                resp_elem = ET.SubElement(respStmt_elem, "resp")
                resp_elem.text = info[key]
                

        editionStmt_elem = ET.SubElement(fileDesc_elem, "editionStmt")
        edition_elem = ET.SubElement(editionStmt_elem, "edition")
        edition_elem.text = "Edition numérique réalisée dans le cadre du projet ANR ArTerm."
        
        publicationStmt_elem = ET.SubElement(fileDesc_elem, "publicationStmt")
        publisher_elemPubli = ET.SubElement(publicationStmt_elem, "publisher")
        publisher_elemPubli.text = "ArTerm"
        pubPlace_elemPubli = ET.SubElement(publicationStmt_elem, "pubPlace")
        pubPlace_elemPubli.text = "Paris"
        date_elemPubli = ET.SubElement(publicationStmt_elem, "date", attrib={"when": "2028"})
        date_elemPubli.text = "2028"
        availability_elem = ET.SubElement(publicationStmt_elem, "availability")
        licence_elem = ET.SubElement(availability_elem, "licence", target="https://creativecommons.org/licenses/by-nc-sa/4.0/")
        licence_elem.text = "ArTerm Copora © 2028 by ArTerm ANR is licensed under CC BY-NC-SA 4.0."

        sourceDesc_elem = ET.SubElement(fileDesc_elem, "sourceDesc")
        
        bibl_elem = ET.SubElement(sourceDesc_elem, "bibl", attrib={"source": info["LienReproduction"]})
        author_elem = ET.SubElement(bibl_elem, "author")
        author_elem.text = info["Prenom"] + " " + info["Nom"]
        isni_elem = ET.SubElement(author_elem, "idno", attrib={"type":"ISNI"})
        isni_elem.text = info["ISNI"]
        title_elemSource = ET.SubElement(bibl_elem, "title", attrib={"xml:lang": "fra"})
        title_elemSource.text = info["Titre"]
        publisher_elem = ET.SubElement(bibl_elem, "publisher")
        publisher_elem.text = info["Editeur"]
        pubPlace_elem = ET.SubElement(bibl_elem, "pubPlace")
        pubPlace_elem.text = info["pubPlace"]
        date_elem = ET.SubElement(bibl_elem, "date", attrib={"when": info["Date"]})
        date_elem.text = info["Date"]
        availability_elem = ET.SubElement(bibl_elem, "availability")
        p_availability_elem = ET.SubElement(availability_elem, "p")
        p_availability_elem.text = info["Droits"]        
        
        xi_elem = ET.SubElement(sourceDesc_elem, "xi:include", href="../IndexOeuvres.xml", xpointer="element(/1/1)")

        encodingDesc_elem = ET.SubElement(root, "encodingDesc")
        projectDesc_elem = ET.SubElement(encodingDesc_elem, "projectDesc")
        p_projectDesc_elem = ET.SubElement(projectDesc_elem, "p")
        p_projectDesc_elem.text = "Le projet ArTerm propose l\'édition numérique d\'un corpus franco-italien de textes de la littérature artistique (XVIe-XVIIe s.), afin d\'examiner les phénomènes de traduction des théories des arts d\'une langue à l\'autre et des transferts linguistiques du lexique esthétique et technique, à l\'aide d\'un Glossaire franco-italien du patrimoine artistique."
        editorialDecl_elem = ET.SubElement(encodingDesc_elem, "editorialDecl")
        correction_elem = ET.SubElement(editorialDecl_elem, "correction", attrib={"status": "low", "method": "silent"})
        p_correction_elem = ET.SubElement(correction_elem, "p")
        p_correction_elem.text = "édition synthétique"
        normalization_elem = ET.SubElement(editorialDecl_elem, "normalization", attrib={"method":"silent"})
        p_normalization_elem = ET.SubElement(normalization_elem, "p")
        p_normalization_elem.text = "Les ambiguïtés entre i, j, u, v ont été corrigées automatiquement ; les espaces autour des ponctuations et accentuations ont été normalisées en suivant les règles de typographies du 20è siècle."
        
        segmentation_elem = ET.SubElement(editorialDecl_elem, "segmentation")
        p_segmentation_elem = ET.SubElement(segmentation_elem, "p")
        p_segmentation_elem.text = "Les paragraphes sont indiqués dans des balises p, enfant de balises div/div1/div2/div3/div4." #Ajouter la phrase générique
        hyphen_elem = ET.SubElement(editorialDecl_elem, "hyphenation", attrib={"eol":"none"})
        stdVals_elem = ET.SubElement(editorialDecl_elem, "stdVals")
        p_stdVals_elem = ET.SubElement(stdVals_elem, "p")
        p_stdVals_elem.text = "Norme ISO-8601 pour les dates"

        profileDesc_elem = ET.SubElement(root, "profileDesc")
        xi_elem = ET.SubElement(profileDesc_elem, "xi:include", href="../IndexPersonnes.xml", xpointer="element(/1/1)")
        xi_elem = ET.SubElement(profileDesc_elem, "xi:include", href="../IndexLieux.xml", xpointer="element(/1/1)")

#Générer les headers
    tree = ET.ElementTree(teiHeader) # création de l'arbre XML
    ET.indent(tree, space="  ", level=0)  # Indentation auto pour une sortie lisible (ici 2 espaces, peut être modifié)
    tree.write(fichier_sortie, encoding="utf-8", xml_declaration=True) # écriture du fichier XML de sortie

    print(f"Header généré avec succès : {fichier_sortie}")
    


## Appel de la fonction generer_header()

Les deux données attendues par cette fonction sont un fichier d'entrée (`header.csv`) qui ne bouge pas, et un fichier de sortie dont le nom peut être modifié si besoin.

In [36]:
generer_header("header_synth.csv", "outputheadersynth.xml")

[{'Nom': 'Armenini', 'Prenom': 'Giovan Battista', 'ISNI': 'https://isni.org/isni/0000000118388240', 'Titre': "De' veri precetti della pittura di M. Gio. Battista Armenini da Faenza libri tre: Ne' quali con bell' ordine d' utili, & buoni avertimenti, per chi desidera in essa farsi con prestezza eccellente; si dimostrano i modi principali del disegnare, & del dipingere, & de fare le Pitture, che si convengono alle conditioni de' luoghi, & delle persone. Opera non solo utile, & necessaria à tutti gli Artefici per cagion del disegno; lume, & fondamento di tutte l' altre arti minori, ma anco à ciascun' altra persona intendente di cosi nobili professione. Al Sereniss. Sig. il Signor Guglielmo Gonzaga Duca di Mantova, di Monferrato, &c. (Ravenna: Francesco Tebaldini, 1586)", 'pubPlace': 'Ravenna', 'Editeur': 'Francesco Tebaldini', 'Date': '1586', 'LienReproduction': 'https://www.digitale-sammlungen.de/en/view/bsb10151942?page=16,17', 'Droits': 'CC', 'Emma Bondoerffer': 'Structuration', 'Anaïs